In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del folder movie_company

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_company")
) 
display(genre_df)

In [0]:
movies_company_df_shema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("companyId", StringType(), True)
])
movies_company_df = (
    spark.read
        .format('csv')
        .schema(movies_company_df_shema)
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_company")
)
display(movies_company_df)
movies_company_df.printSchema()

In [0]:
movies_company_df = (
    add_ingestion_date(movies_company_df)
    .select(
        col("movieId").alias("movie_id"),
        col("companyId").alias("company_id"),
        lit(v_env).alias("env"),
        lit(v_file_date).alias("file_date")
    )
)
display(movies_company_df)

In [0]:
overwrite_partitioned_table("movie_silver", "movies_companies", "file_date", v_file_date)

In [0]:
(
    movies_company_df
    .write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable("movie_silver.movies_companies")
)

In [0]:
dbutils.notebook.exit("Success")